# Part 4: Paged Attention (Problems 025–033)

**The problem with a flat KV cache:**

With a simple KV cache, each sequence pre-allocates a contiguous buffer large enough for `max_seq_len` tokens. This leads to severe memory fragmentation:

```
Naive allocation (max_seq_len=512):
  Request A: 50 tokens used  → 462 slots WASTED
  Request B: 200 tokens used → 312 slots WASTED
  Request C: 512 tokens used → 0 slots wasted
  Request D: REJECTED (no more contiguous memory)
```

**Paged Attention (vLLM)** solves this by dividing the KV cache into fixed-size **pages** (blocks), and allocating pages to sequences on demand — just like virtual memory in an OS:

```
Paged allocation (page_size=16):
  Page pool: [P0, P1, P2, ..., P255]  (256 pages × 16 slots each = 4096 total)
  Request A: needs 4 pages  → [P0, P1, P2, P3]
  Request B: needs 13 pages → [P4..P16]
  Request C: needs 32 pages → [P17..P48]
  Request D: allocate from remaining free pages
```

## Cell 1: Simulate naive allocation, show wasted memory

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Make sure the project root is on sys.path so solutions/ is importable
project_root = Path('__file__').parent.parent if '__file__' in dir() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Also try the current directory's parent
for p in [Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / 'solutions').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break


In [ ]:
import importlib

try:
    _m = importlib.import_module("solutions.025_understand_memory_fragmentation")
    compute_fragmentation = getattr(_m, "compute_fragmentation", None) or getattr(_m, "understand_memory_fragmentation", None)
    if compute_fragmentation is None:
        funcs = [v for k, v in vars(_m).items() if callable(v) and not k.startswith('_')]
        compute_fragmentation = funcs[0] if funcs else None
except Exception:
    print("Solve problem 025 first:")
    print("  cp problems/025_understand_memory_fragmentation.py solutions/025_understand_memory_fragmentation.py")
    compute_fragmentation = None

# Simulate naive allocation manually
import random
random.seed(42)

max_seq_len = 512
n_requests = 10

# Actual lengths follow a realistic distribution (shorter is more common)
actual_lengths = [random.randint(10, 300) for _ in range(n_requests)]

total_allocated = n_requests * max_seq_len
total_used = sum(actual_lengths)
total_wasted = total_allocated - total_used
fragmentation_pct = 100 * total_wasted / total_allocated

print(f"Naive KV cache allocation (max_seq_len={max_seq_len}):")
print()
print(f"  {'Request':>10}  {'Allocated':>10}  {'Used':>6}  {'Wasted':>8}")
print(f"  {'-'*10}  {'-'*10}  {'-'*6}  {'-'*8}")
for i, used in enumerate(actual_lengths):
    wasted = max_seq_len - used
    bar_used  = "#" * (used  * 30 // max_seq_len)
    bar_waste = "." * (wasted * 30 // max_seq_len)
    print(f"  Request {i+1:>2}:  {max_seq_len:>10}  {used:>6}  {wasted:>8}  [{bar_used}{bar_waste}]")

print()
print(f"  Total allocated : {total_allocated:>6} slots")
print(f"  Total used      : {total_used:>6} slots")
print(f"  Total wasted    : {total_wasted:>6} slots ({fragmentation_pct:.1f}% fragmentation)")

## Cell 2: Create page pool, allocate pages to sequences

In [ ]:
import importlib

try:
    _m = importlib.import_module("solutions.027_allocate_page_pool")
    allocate_page_pool = _m.allocate_page_pool
except Exception:
    print("Solve problem 027 first:")
    print("  cp problems/027_allocate_page_pool.py solutions/027_allocate_page_pool.py")
    allocate_page_pool = None

try:
    _m = importlib.import_module("solutions.028_assign_page_to_sequence")
    assign_page_to_sequence = _m.assign_page_to_sequence
except Exception:
    print("Solve problem 028 first:")
    print("  cp problems/028_assign_page_to_sequence.py solutions/028_assign_page_to_sequence.py")
    assign_page_to_sequence = None

if allocate_page_pool is not None:
    page_size = 16
    n_pages = 32
    n_layers = 4
    n_heads = 4
    d_head = 16

    pool = allocate_page_pool(
        n_pages=n_pages,
        page_size=page_size,
        n_layers=n_layers,
        n_heads=n_heads,
        d_head=d_head,
    )

    print(f"Page pool created:")
    print(f"  n_pages  = {n_pages}")
    print(f"  page_size= {page_size} tokens per page")
    print(f"  Total KV slots: {n_pages * page_size}")
    print(f"  Pool type: {type(pool).__name__}")

    if assign_page_to_sequence is not None:
        print()
        print("Assigning pages to sequences:")
        block_table = {}  # seq_id -> list of page indices

        for seq_id, n_tokens in [("seq_A", 30), ("seq_B", 50), ("seq_C", 18)]:
            n_pages_needed = (n_tokens + page_size - 1) // page_size
            block_table[seq_id] = []
            for _ in range(n_pages_needed):
                page_idx = assign_page_to_sequence(pool, seq_id, block_table)
                block_table[seq_id].append(page_idx)
            print(f"  {seq_id}: {n_tokens} tokens → "
                  f"{n_pages_needed} pages → block_table={block_table[seq_id]}")

## Cell 3: Write KV to pages, read back, verify correctness

In [ ]:
import importlib
import torch

try:
    _m = importlib.import_module("solutions.029_write_kv_to_page")
    write_kv_to_page = _m.write_kv_to_page
except Exception:
    print("Solve problem 029 first:")
    print("  cp problems/029_write_kv_to_page.py solutions/029_write_kv_to_page.py")
    write_kv_to_page = None

try:
    _m = importlib.import_module("solutions.030_read_kv_via_block_table")
    read_kv_via_block_table = _m.read_kv_via_block_table
except Exception:
    print("Solve problem 030 first:")
    print("  cp problems/030_read_kv_via_block_table.py solutions/030_read_kv_via_block_table.py")
    read_kv_via_block_table = None

torch.manual_seed(99)

page_size = 4
n_heads = 2
d_head = 8
n_pages = 8

# Pages: shape [n_pages, page_size, n_heads, d_head]
k_pages = torch.zeros(n_pages, page_size, n_heads, d_head)
v_pages = torch.zeros(n_pages, page_size, n_heads, d_head)

# Write some known values
test_k = torch.randn(page_size, n_heads, d_head)
test_v = torch.randn(page_size, n_heads, d_head)
target_page = 3

if write_kv_to_page is not None:
    k_pages, v_pages = write_kv_to_page(
        k_pages=k_pages, v_pages=v_pages,
        page_idx=target_page,
        k=test_k, v=test_v
    )
    print(f"Wrote KV to page {target_page}")

    if read_kv_via_block_table is not None:
        block_table = [target_page]  # sequence uses only page 3
        k_read, v_read = read_kv_via_block_table(
            k_pages=k_pages, v_pages=v_pages,
            block_table=block_table
        )
        match_k = torch.allclose(k_read, test_k)
        match_v = torch.allclose(v_read, test_v)
        print(f"Read back K matches: {match_k}")
        print(f"Read back V matches: {match_v}")
        if match_k and match_v:
            print("KV round-trip through pages: PASSED")
        else:
            print("KV round-trip through pages: FAILED — check your implementation")
else:
    print("Complete problems 029 and 030 to test KV page read/write.")

## Cell 4: Simulate freeing pages on completion

In [ ]:
import importlib

try:
    _m = importlib.import_module("solutions.031_free_pages_on_completion")
    free_pages_on_completion = _m.free_pages_on_completion
except Exception:
    print("Solve problem 031 first:")
    print("  cp problems/031_free_pages_on_completion.py solutions/031_free_pages_on_completion.py")
    free_pages_on_completion = None

# Simulate a page pool lifecycle
n_pages = 16
free_pages = list(range(n_pages))
block_tables = {}

def allocate_pages(seq_id, n):
    pages = free_pages[:n]
    del free_pages[:n]
    block_tables[seq_id] = pages
    return pages

# Allocate pages for 3 sequences
allocate_pages("A", 3)
allocate_pages("B", 5)
allocate_pages("C", 4)

print(f"After allocating for 3 sequences:")
print(f"  Free pages    : {free_pages}  ({len(free_pages)} remaining)")
print(f"  Seq A pages   : {block_tables['A']}")
print(f"  Seq B pages   : {block_tables['B']}")
print(f"  Seq C pages   : {block_tables['C']}")

if free_pages_on_completion is not None:
    # Sequence B completes
    free_pages_list, block_tables = free_pages_on_completion(
        seq_id="B",
        block_tables=block_tables,
        free_pages=free_pages,
    )
    print()
    print("After sequence B completes:")
    print(f"  Free pages    : {free_pages_list}  ({len(free_pages_list)} available)")
    print(f"  Block tables  : {block_tables}")
    assert "B" not in block_tables, "Sequence B should be removed from block table"
    print("Pages successfully returned to the pool!")
else:
    print()
    print("Implement problem 031 to see pages returned to the pool.")

## Cell 5: Memory utilisation comparison — naive vs paged

In [ ]:
import importlib
import random
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

try:
    _m = importlib.import_module("solutions.033_benchmark_memory_utilization")
    benchmark_memory_utilization = _m.benchmark_memory_utilization
except Exception:
    print("Solve problem 033 first:")
    print("  cp problems/033_benchmark_memory_utilization.py solutions/033_benchmark_memory_utilization.py")
    benchmark_memory_utilization = None

random.seed(123)
n_requests = 20
max_seq_len = 256
page_size = 16
lengths = [random.randint(8, max_seq_len) for _ in range(n_requests)]

# Naive: always allocate max_seq_len per request
naive_allocated = [max_seq_len] * n_requests
naive_used = lengths
naive_efficiency = [u/a*100 for u, a in zip(naive_used, naive_allocated)]

# Paged: allocate ceil(length/page_size) pages
import math
paged_allocated = [math.ceil(l/page_size)*page_size for l in lengths]
paged_used = lengths
paged_efficiency = [u/a*100 for u, a in zip(paged_used, paged_allocated)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(n_requests)
axes[0].bar(x, naive_allocated, color="lightcoral", label="Allocated")
axes[0].bar(x, naive_used, color="steelblue", label="Actually Used")
axes[0].set_title(f"Naive allocation\n(avg efficiency: {sum(naive_efficiency)/len(naive_efficiency):.1f}%)")
axes[0].set_xlabel("Request")
axes[0].set_ylabel("KV Slots")
axes[0].legend()

axes[1].bar(x, paged_allocated, color="lightcoral", label="Allocated")
axes[1].bar(x, paged_used, color="steelblue", label="Actually Used")
axes[1].set_title(f"Paged allocation (page_size={page_size})\n(avg efficiency: {sum(paged_efficiency)/len(paged_efficiency):.1f}%)")
axes[1].set_xlabel("Request")
axes[1].set_ylabel("KV Slots")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/paged_attention_memory.png", dpi=100)
plt.show()

print(f"Total naive waste : {sum(naive_allocated) - sum(naive_used)} slots")
print(f"Total paged waste : {sum(paged_allocated) - sum(paged_used)} slots")
print(f"Waste reduction   : {(sum(naive_allocated)-sum(naive_used))/(sum(paged_allocated)-sum(paged_used)):.1f}x less waste with paging")